# Reward Models

Companion notebook for the [Reward Models lesson](https://ml-viz-ruby.vercel.app/courses/fine-tuning-alignment/03-reward-models).

**The idea in one sentence.** You can't write down a reward for "helpful" — so you
*learn* one from human **preferences**: a reward model trained with the
**Bradley–Terry** loss to score a chosen response higher than a rejected one, which
then drives RLHF.

$$\mathcal{L} = -\log \sigma\big(r(x, y_{\text{chosen}}) - r(x, y_{\text{rejected}})\big)$$

The crucial property: the loss depends only on the **difference** of rewards, so the
reward is defined only up to an additive constant — its *absolute* scale is
meaningless, only comparisons matter.

We build the BT loss from scratch, **gradient-check it, confirm the reward is
shift-invariant, and that it generalises to held-out pairs**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
rng = np.random.default_rng(0)

## A toy reward model

Real reward models are LM trunks with a scalar head. We strip everything down to the head: features $\varphi(x, y) \in \mathbb{R}^d$ (think "the last hidden state") and a linear reward

$$
r_\theta(x, y) = w^\top \varphi(x, y).
$$

We fake the features for chosen and rejected responses by sampling them from two Gaussians whose centres differ in a true latent direction $w^\star$. The reward model has to recover that direction from pairwise comparisons alone.

In [ ]:
d = 16            # feature dim
n_pairs = 1000    # training preference pairs
n_test = 500      # held-out pairs

# Ground-truth reward direction (unknown to the model).
w_star = rng.normal(0, 1, size=d)
w_star /= np.linalg.norm(w_star)

def sample_pair(n):
    """Synthesise (chosen, rejected) feature pairs whose true reward gap
    is positive on average but noisy on individual pairs."""
    # Shared 'prompt' features: random direction.
    base = rng.normal(0, 1, size=(n, d))
    # Chosen responses lean a bit further along w_star, rejected lean a bit away.
    chosen = base + 0.6 * w_star + rng.normal(0, 0.4, size=(n, d))
    rejected = base - 0.6 * w_star + rng.normal(0, 0.4, size=(n, d))
    # Flip 10% of labels to model labeller noise / disagreement.
    flip = rng.random(n) < 0.10
    c, r = chosen.copy(), rejected.copy()
    c[flip], r[flip] = rejected[flip], chosen[flip]
    return c, r

phi_c, phi_r = sample_pair(n_pairs)
phi_c_test, phi_r_test = sample_pair(n_test)
print(f'chosen features:   {phi_c.shape}')
print(f'rejected features: {phi_r.shape}')

## Bradley–Terry loss and its gradient

The per-pair loss is

$$
\ell(w) = -\log \sigma\!\big(w^\top \varphi_c - w^\top \varphi_r\big) = -\log \sigma(w^\top \Delta\varphi),
$$

where $\Delta\varphi = \varphi_c - \varphi_r$. The gradient is clean:

$$
\nabla_w \ell = -\big(1 - \sigma(w^\top \Delta\varphi)\big)\, \Delta\varphi.
$$

Intuition: if the model is already confident the chosen wins ($\sigma \approx 1$), the gradient is small. The hard, ambiguous pairs do most of the learning. Same trick that powers logistic regression.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def bt_loss(w, phi_c, phi_r):
    dphi = phi_c - phi_r
    z = dphi @ w
    # log-sigmoid, stable form: -log(1+exp(-z)) = -softplus(-z)
    return np.mean(np.log1p(np.exp(-z)))

def bt_grad(w, phi_c, phi_r):
    dphi = phi_c - phi_r
    z = dphi @ w
    g = -(1 - sigmoid(z))[:, None] * dphi
    return g.mean(axis=0)

def pairwise_accuracy(w, phi_c, phi_r):
    return float(np.mean((phi_c - phi_r) @ w > 0))

### Validate: the Bradley–Terry gradient and its shift-invariance

Two checks. First, the analytic `bt_grad` must match finite differences. Second, the
loss depends only on $r_c - r_r$, so *adding a constant reward direction that affects
chosen and rejected equally* leaves the loss unchanged — the reward has no meaningful
zero point.

In [ ]:
# 1. gradient check
w_gc = rng.normal(0, 0.5, size=d)
g = bt_grad(w_gc, phi_c, phi_r)
eps = 1e-6
num = np.zeros(d)
for j in range(d):
    wp = w_gc.copy(); wp[j] += eps
    wm = w_gc.copy(); wm[j] -= eps
    num[j] = (bt_loss(wp, phi_c, phi_r) - bt_loss(wm, phi_c, phi_r)) / (2*eps)
print(f'max |analytic - numeric| gradient: {np.abs(g - num).max():.2e}')
assert np.allclose(g, num, atol=1e-6), 'BT gradient must match finite differences'

# 2. shift invariance: add the SAME feature offset to chosen and rejected -> margin unchanged
offset = rng.normal(0, 1, size=d)
loss_shifted = bt_loss(w_gc, phi_c + offset, phi_r + offset)
loss_plain = bt_loss(w_gc, phi_c, phi_r)
print(f'loss unchanged by a common shift: {loss_plain:.6f} vs {loss_shifted:.6f}')
assert np.isclose(loss_plain, loss_shifted), 'the reward only depends on chosen-minus-rejected'
print('\n✅ BT gradient is correct, and the reward is defined only up to comparisons')

## Train with vanilla gradient descent

We initialise $w$ at zero, which means every pair starts at margin 0 → $\sigma(0) = 0.5$ → loss $\log 2 \approx 0.693$. That's our random baseline.

In [ ]:
w = np.zeros(d)
losses = []
test_accs = []
for step in range(400):
    losses.append(bt_loss(w, phi_c, phi_r))
    test_accs.append(pairwise_accuracy(w, phi_c_test, phi_r_test))
    w -= 0.5 * bt_grad(w, phi_c, phi_r)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(losses, color='#6366f1')
axes[0].axhline(np.log(2), color='#94a3b8', linestyle='--', label='random (log 2)')
axes[0].set_xlabel('step'); axes[0].set_ylabel('BT loss')
axes[0].set_title('Bradley–Terry loss drops below the log-2 floor')
axes[0].legend()

axes[1].plot(test_accs, color='#14b8a6')
axes[1].axhline(0.5, color='#94a3b8', linestyle='--', label='random')
axes[1].set_xlabel('step'); axes[1].set_ylabel('held-out pairwise accuracy')
axes[1].set_title('Held-out pairwise accuracy')
axes[1].set_ylim(0.45, 1.0)
axes[1].legend()
plt.tight_layout(); plt.show()
print(f'final BT loss        : {losses[-1]:.4f}')
print(f'final test accuracy  : {test_accs[-1]:.3f}')

### Validate: the reward model generalises to held-out preferences

A reward model is only useful if it ranks *unseen* pairs correctly. We check the
trained model's pairwise accuracy on the held-out set beats chance substantially, and
that the BT loss dropped below the $\log 2$ random-guess floor.

In [ ]:
test_acc = pairwise_accuracy(w, phi_c_test, phi_r_test)
print(f'held-out pairwise accuracy: {test_acc:.3f}  (chance = 0.5)')
print(f'final BT loss: {losses[-1]:.3f}  (random-guess floor log 2 = {np.log(2):.3f})')
assert test_acc > 0.75, 'the trained reward model should rank held-out pairs well'
assert losses[-1] < np.log(2), 'the loss should drop below the random-guess floor'
print('\n✅ the reward model generalises: it ranks unseen chosen>rejected pairs correctly')

## Reward histograms: chosen vs rejected

A well-trained RM separates the two distributions. Note the **overlap** at the boundary — that's where the 10 % label noise we baked in lives. A perfectly clean RM is usually a leak.

In [ ]:
r_chosen = phi_c_test @ w
r_rejected = phi_r_test @ w

plt.hist(r_rejected, bins=30, alpha=0.7, color='#f43f5e', label='rejected', density=True)
plt.hist(r_chosen, bins=30, alpha=0.7, color='#14b8a6', label='chosen', density=True)
plt.xlabel('reward r(x, y)')
plt.ylabel('density')
plt.title('Held-out reward distribution: chosen lifts above rejected')
plt.legend()
plt.show()
print(f'mean reward (chosen)   : {r_chosen.mean():+.3f}')
print(f'mean reward (rejected) : {r_rejected.mean():+.3f}')
print(f'mean margin            : {(r_chosen - r_rejected).mean():+.3f}')

## Sanity check: shifting all rewards changes nothing

The BT loss only sees the **margin** — add the same constant to every reward and the loss is identical. We verify.

In [ ]:
loss_now = bt_loss(w, phi_c, phi_r)
# Equivalent: add a constant to every reward (i.e. a constant bias) doesn't show
# up in (r_c - r_r). We bake that into the loss explicitly to make the point.

def bt_loss_with_bias(w, b, phi_c, phi_r):
    z = (phi_c - phi_r) @ w + (b - b)   # bias cancels in the margin
    return np.mean(np.log1p(np.exp(-z)))

for bias in [-5.0, -1.0, 0.0, 3.0, 10.0]:
    print(f'  bias = {bias:+5.1f}  →  loss = {bt_loss_with_bias(w, bias, phi_c, phi_r):.6f}')
print(f'  reference (no bias)  loss = {loss_now:.6f}')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **reward hacking** | optimising the RM too hard finds OOD high-reward garbage (demo) → KL-regularize |
| **absolute reward is meaningless** | only differences matter; don't threshold raw reward values |
| **preference noise & annotator disagreement** | the ceiling is set by label quality |
| **distribution shift** | as the policy improves, its outputs leave the RM's training distribution |
| **length/style bias** | RMs latch onto verbosity or formatting instead of substance |

Demo: the reward model confidently over-scores out-of-distribution noise — the seed
of reward hacking.

In [ ]:
# The central danger with a learned reward: REWARD HACKING. The RM is only accurate
# near the data it saw; push far along its gradient (as RL will) and you reach
# out-of-distribution inputs where the score is high but the true quality is not.
# We show the RM confidently scores pure-noise 'responses' it never saw.
noise_responses = rng.normal(0, 3, size=(500, d))     # far-from-distribution inputs
noise_rewards = noise_responses @ w
real_rewards = phi_c_test @ w
print(f'mean RM score on real chosen responses : {real_rewards.mean():+.2f}')
print(f'max  RM score on random noise          : {noise_rewards.max():+.2f}  (higher than most real ones!)')
frac_beating = (noise_rewards > np.median(real_rewards)).mean()
print(f'fraction of noise scoring above the median real response: {frac_beating:.0%}')
print('\nAn optimizer will EXPLOIT these OOD high-reward points -> KL-regularize toward a')
print('reference policy (next lesson) so RLHF cannot wander into the RM blind spots.')

## ✏️ Your turn

Write `pairwise_accuracy_split(w, phi_c, phi_r)` that returns the **held-out pairwise accuracy** — the fraction of test pairs for which the model gives the chosen response a strictly higher reward than the rejected one. This is the standard RM evaluation metric.

A correctly-trained RM on this dataset should reach roughly 0.85 or better. Random guessing is 0.5.

In [ ]:
def pairwise_accuracy_split(w, phi_c, phi_r):
    # TODO(you): return the fraction of pairs where r(chosen) > r(rejected).
    # Hint: the reward is r = phi @ w; compute the margin and check its sign.
    return 0.0

acc = pairwise_accuracy_split(w, phi_c_test, phi_r_test)
assert 0.0 <= acc <= 1.0, 'must return a probability'
assert acc > 0.75, f'trained RM should beat 0.75 on test pairs, got {acc:.3f}'
# Sanity: random w gives roughly chance.
assert abs(pairwise_accuracy_split(np.zeros(d), phi_c_test, phi_r_test) - 0.5) < 0.1
print(f'pairwise accuracy: {acc:.3f}')
print('passed ✓')

<details><summary>Solution</summary>

```python
def pairwise_accuracy_split(w, phi_c, phi_r):
    margin = (phi_c - phi_r) @ w
    return float(np.mean(margin > 0))
```

Equivalently: `float(np.mean((phi_c @ w) > (phi_r @ w)))`. The two are identical because subtraction is linear.

</details>

## Key takeaways

- **Reward models learn "good" from preferences** via the Bradley–Terry loss
  $-\log\sigma(r_c - r_r)$ (we gradient-checked it).
- **The reward is relative, not absolute:** it depends only on differences, so its
  zero point is meaningless (verified shift-invariance).
- **It generalises** to held-out pairs (verified >0.75 accuracy) — that's what makes
  it useful as an RLHF signal.
- **Reward hacking is the central risk:** the RM is only trustworthy near its training
  distribution; RL will exploit OOD high-reward points (demo) — hence the KL leash in
  the next lesson.